# Reinforcement Learning From AI Feedback (RLAIF) with Serverless customization on SageMaker AI

## Lab 4.4 - Evaluation

In this notebook, we are going to evaluate the fine-tuned Qwen 2.5 - 7B Instruct model, using the eval dataset created in the first notebook

---

## Prerequisites
### AWS Access Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

project_prefix = "humanlike-rlaif"
base_model_jumpstart_id = "huggingface-llm-qwen2-5-7b-instruct"
base_model_shortname = "qwen25-7b"

### Get Model Package Group

In [ ]:
from sagemaker.core.resources import ModelPackageGroup

model_package_group_name=f"{project_prefix}-{base_model_shortname}"
model_package_version = "1"

mpg = ModelPackageGroup.get(model_package_group_name)
fine_tuned_model_package_group_arn = mpg.model_package_group_arn
print(fine_tuned_model_package_group_arn)

fine_tuned_model_package_arn = f"{mpg.model_package_group_arn.replace('model-package-group', 'model-package', 1)}/{model_package_version}"
print(fine_tuned_model_package_arn)

bucket_name = sess.default_bucket()
s3_output_path = f"s3://{bucket_name}/{project_prefix}-{base_model_shortname}/eval/".format(bucket_name)
print(s3_output_path)

### Configure LLM-as-Judge Evaluation

We set up two parallel evaluation jobs using Amazon Nova Pro as the judge model — one for the base model and one for our RLAIF fine-tuned model. Both use a custom "human-like alignment" metric that scores responses on naturalness, tone, and conversational flow against ground truth references. Running identical evaluations on both models lets us quantify the impact of RLAIF fine-tuning.

In [ ]:
import json
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.train.evaluate import LLMAsJudgeEvaluator

evaluator_model_id = "amazon.nova-pro-v1:0"
eval_dataset = DataSet.get(f"{project_prefix}-eval")

custom_metrics_list = [
    {
        "customMetricDefinition": {
            "name": "human-like-alignment",
            "instructions": (
                "You are an expert at evaluating language model outputs and "
                "determining which responses sound more natural and human-like. "
                "Be extremely critical and look at it very thoroughly.\n\n"
                "You may use the ground truth response as a reference of what "
                "a human-like answer should contain.\n\n"
                "Focus on these aspects when evaluating:\n"
                "- Human-like reasoning and explanations\n"
                "- Emotional intelligence and empathy where relevant\n"
                "- Avoidance of overly rigid or mechanical language\n"
                "- Natural conversation flow\n"
                "- Appropriate level of formality\n\n"
                "Here is the actual task:\n"
                "Task: {{prompt}}\n"
                "Ground Truth Response: {{ground_truth}}\n"
                "Candidate Response: {{prediction}}"
            ),
            "ratingScale": [
                {
                    "definition": "No part of the response sounds like a human-like response.",
                    "value": {
                        "floatValue": 0
                    }
                },
                {
                    "definition": "Roughly half of the response is a human-like response.",
                    "value": {
                        "floatValue": 1
                    }
                },
                {
                    "definition": "Every piece of the response sounds like a human-like response.",
                    "value": {
                        "floatValue": 2
                    }
                }
            ]
        }
    }
]

custom_metrics_json = json.dumps(custom_metrics_list)

evaluator = LLMAsJudgeEvaluator(
    model=fine_tuned_model_package_arn,
    model_package_group=fine_tuned_model_package_group_arn,
    evaluator_model=evaluator_model_id,
    dataset=eval_dataset.arn,
    custom_metrics=custom_metrics_json,
    s3_output_path=s3_output_path,  # Required
    evaluate_base_model=True # This will automatically trigger another evaluation, using the base (uncustomized) model
)

### Start the Evaluation jobs

In [ ]:
execution = evaluator.evaluate()

---

## Analyze Evaluation results
### Download the evaluation output from S3

In [ ]:
import os

execution_id = execution.arn.split("/")[-1]
output_path = execution.s3_output_path

print(f"Execution ID: {execution_id}")
print(f"S3 output path: {output_path}")

os.makedirs("./eval_results", exist_ok=True)

s3 = boto3.client("s3", region_name=boto3.Session().region_name)

bucket = output_path.replace("s3://", "").split("/")[0]
prefix = "/".join(output_path.replace("s3://", "").split("/")[1:])

base_key = None
custom_key = None

for obj in s3.list_objects_v2(Bucket=bucket, Prefix=prefix)["Contents"]:
    key = obj["Key"]
    if execution_id in key and "_output.jsonl" in key and "/datasets/" in key:
        if "base-llmaj-eval" in key:
            base_key = key
        elif "custom-llmaj-eval" in key:
            custom_key = key

print(f"Base: {base_key}")
print(f"Custom: {custom_key}")

s3.download_file(bucket, base_key, "./eval_results/base_eval.jsonl")
s3.download_file(bucket, custom_key, "./eval_results/custom_eval.jsonl")

print("Downloaded to ./eval_results/")


### Parse and display the evaluation results

In [ ]:
# Analyze results
def get_score(result):
    scores = result.get('automatedEvaluationResult', {}).get('scores', [])
    for score in scores:
        if score.get('metricName') == 'human-like-alignment':
            return score.get('result')
    return None

with open('./eval_results/base_eval.jsonl') as f:
    base_results = [json.loads(line) for line in f]

with open('./eval_results/custom_eval.jsonl') as f:
    custom_results = [json.loads(line) for line in f]

base_scores = [get_score(r) for r in base_results if get_score(r) is not None]
custom_scores = [get_score(r) for r in custom_results if get_score(r) is not None]

# Calculate metrics (scores are 0.0, 1.0, 2.0)
base_avg = sum(base_scores) / len(base_scores) / 2 * 100 if base_scores else 0
custom_avg = sum(custom_scores) / len(custom_scores) / 2 * 100 if custom_scores else 0

# Distribution
def get_distribution(scores):
    completely = sum(1 for s in scores if s == 2.0)
    somewhat = sum(1 for s in scores if s == 1.0)
    not_at_all = sum(1 for s in scores if s == 0.0)
    return completely, somewhat, not_at_all

base_dist = get_distribution(base_scores)
custom_dist = get_distribution(custom_scores)

print("\n" + "=" * 80)
print("EVALUATION RESULTS")
print("=" * 80)
print(f"\nBase Model:")
print(f"  Average Score: {base_avg:.1f}%")
print(f"  Valid Samples: {len(base_scores)}/{len(base_results)}")
print(f"  Distribution:")
print(f"    - Completely human-like (2.0): {base_dist[0]} ({base_dist[0]/len(base_scores)*100:.1f}%)")
print(f"    - Somewhat human-like (1.0): {base_dist[1]} ({base_dist[1]/len(base_scores)*100:.1f}%)")
print(f"    - Not at all human-like (0.0): {base_dist[2]} ({base_dist[2]/len(base_scores)*100:.1f}%)")

print(f"\nFine-tuned Model:")
print(f"  Average Score: {custom_avg:.1f}%")
print(f"  Valid Samples: {len(custom_scores)}/{len(custom_results)}")
print(f"  Distribution:")
print(f"    - Completely human-like (2.0): {custom_dist[0]} ({custom_dist[0]/len(custom_scores)*100:.1f}%)")
print(f"    - Somewhat human-like (1.0): {custom_dist[1]} ({custom_dist[1]/len(custom_scores)*100:.1f}%)")
print(f"    - Not at all human-like (0.0): {custom_dist[2]} ({custom_dist[2]/len(custom_scores)*100:.1f}%)")

print(f"\n{'='*80}")
print(f"IMPROVEMENT: {custom_avg - base_avg:+.1f}%")
print(f"{'='*80}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Distribution comparison
categories = ['Not at all\n(0.0)', 'Somewhat\n(1.0)', 'Completely\n(2.0)']
base_counts = [base_dist[2], base_dist[1], base_dist[0]]
custom_counts = [custom_dist[2], custom_dist[1], custom_dist[0]]

x = np.arange(len(categories))
width = 0.35

bars1 = ax1.bar(x - width/2, base_counts, width, label='Base Model', alpha=0.8)
bars2 = ax1.bar(x + width/2, custom_counts, width, label='Fine-tuned Model', alpha=0.8)

ax1.set_xlabel('Human-like Alignment Score')
ax1.set_ylabel('Number of Samples')
ax1.set_title('Distribution of Scores')
ax1.set_xticks(x)
ax1.set_xticklabels(categories)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Average score comparison
models = ['Base Model', 'Fine-tuned Model']
averages = [base_avg, custom_avg]
colors = ['#1f77b4', '#ff7f0e']

bars = ax2.bar(models, averages, color=colors, alpha=0.8)
ax2.set_ylabel('Average Score (%)')
ax2.set_title('Average Human-like Alignment Score')
ax2.set_ylim(0, 100)
ax2.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}%',
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()